In [50]:
import pandas as pd


In [51]:
df = pd.read_csv("/content/Kaggle_Ecommerce Data.csv")

In [52]:
df.head()

/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,order_id,customer_id,product_id,category,price,discount,quantity,payment_method,order_date,delivered_date,region,returned,request_date,return_reason,total_amount,shipping_cost,profit_margin,customer_age,customer_gender
0,O100000,C17270,P234890,Home,164.08,0.15,1,Credit Card,23/12/2023,27/12/2023,West,No,NaN,NaN,139.47,7.88,31.17,60,Female
1,O100001,C17603,P228204,Grocery,24.73,0.00,1,Credit Card,3/4/2025,9/4/2025,South,No,NaN,NaN,24.73,4.60,-2.62,37,Male
2,O100002,C10860,P213892,Electronics,175.58,0.05,1,Credit Card,8/10/2024,12/10/2024,North,No,NaN,NaN,166.80,6.58,13.44,34,Male
3,O100003,C15390,P208689,Electronics,63.67,0.00,1,UPI,14/9/2024,20/9/2024,South,No,NaN,NaN,63.67,5.50,2.14,21,Female
4,O100004,C15226,P228063,Home,16.33,0.15,1,COD,21/12/2024,27/12/2024,East,No,NaN,NaN,13.88,2.74,1.15,39,Male


In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34500 entries, 0 to 34499
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         34500 non-null  object 
 1   customer_id      34500 non-null  object 
 2   product_id       34500 non-null  object 
 3   category         34500 non-null  object 
 4   price            34500 non-null  float64
 5   discount         34500 non-null  float64
 6   quantity         34500 non-null  int64  
 7   payment_method   34500 non-null  object 
 8   order_date       34500 non-null  object 
 9   delivered_date   34500 non-null  object 
 10  region           34500 non-null  object 
 11  returned         34500 non-null  object 
 12  request_date     1903 non-null   object 
 13  return_reason    1903 non-null   object 
 14  total_amount     34500 non-null  float64
 15  shipping_cost    34500 non-null  float64
 16  profit_margin    34500 non-null  float64
 17  customer_age

In [72]:
df.isnull().sum()

,0
order_id,0
customer_id,0
product_id,0
category,0
price,0
discount,0
quantity,0
payment_method,0
order_date,0
delivered_date,0


In [54]:
print(df["order_id"].duplicated().sum())

0


In [55]:
print((df["delivered_date"] < df["order_date"]).sum())

8088


In [56]:
print(df[["request_date", "return_reason"]].isna().sum())

request_date     32597
return_reason    32597
dtype: int64


In [71]:
df['order_date'] = pd.to_datetime(df['order_date'], format='%d/%m/%Y')
df["year_month"] = df["order_date"].dt.to_period("M").astype(str)

In [58]:
for col in ["order_date", "delivered_date", "request_date"]:
    df[col] = pd.to_datetime(df[col], format="%d/%m/%Y")

In [59]:
df["is_returned"] = (df["returned"] == "Yes").astype(int)

In [60]:
df["net_sales"] = df["total_amount"] * (1 - df["is_returned"])

In [61]:
def summary(col):
    return df.groupby(col).agg(
        orders=("order_id", "count"),
        sales=("total_amount", "sum"),
        profit=("profit_margin", "sum"),
        return_rate=("is_returned", "mean")
    ).round(3)

In [62]:
print(summary("category"))

             orders       sales     profit  return_rate
category                                               
Beauty         4103   153019.38   49196.59        0.038
Electronics    6180  3319206.50  344371.77        0.073
Fashion        6254   471545.80  128814.65        0.083
Grocery        4058    82000.51   -9187.96        0.013
Home           5487  1077681.52  262633.70        0.056
Sports         4171   629825.54  160521.41        0.049
Toys           4247   132013.80   33669.25        0.049


In [63]:
print(summary("region"))

         orders       sales     profit  return_rate
region                                             
Central    5632   940503.38  158023.69        0.051
East       6904  1176334.75  195035.24        0.059
North      7572  1264008.35  208493.58        0.054
South      7584  1298096.07  211081.17        0.057
West       6808  1186350.50  197385.73        0.054


In [64]:
print(summary("payment_method"))

                orders       sales     profit  return_rate
payment_method                                            
COD               4160   715571.90  116192.61        0.051
Credit Card      12170  2056787.40  341894.20        0.055
Debit Card        8505  1460210.97  241063.38        0.057
PayPal            3444   576523.32   95414.72        0.058
UPI               4156   713642.96  117103.32        0.054
Wallet            2065   342556.50   58351.18        0.054


In [65]:
print(df["return_reason"].value_counts())

return_reason
Not as described      490
No longer needed      481
Defective             465
Missing/Wrong item    439
Slow delivery          28
Name: count, dtype: int64


In [66]:
df.to_csv("cleaned_ecommerce.csv", index=False)

In [67]:
# 1. Outliers (bade orders)
q1, q3 = df["total_amount"].quantile([0.25, 0.75])
outliers = df[df["total_amount"] > q3 + 1.5 * (q3 - q1)]
print(len(outliers), outliers["total_amount"].sum() / df["total_amount"].sum())
print(outliers["category"].value_counts())





3792 0.5733163170764332
category
Electronics    2698
Home            670
Sports          310
Fashion         103
Beauty            7
Grocery           2
Toys              2
Name: count, dtype: int64


In [68]:
df["delivery_days"] = (df["delivered_date"] - df["order_date"]).dt.days
df["is_returned"] = (df["returned"] == "Yes").astype(int)

In [69]:

# 3. Customer groups ka return rate
df["age_group"] = pd.cut(df["customer_age"], [17, 25, 35, 45, 55, 70],
                         labels=["18-25", "26-35", "36-45", "46-55", "56+"])
for col in ["age_group", "customer_gender", "payment_method", "quantity"]:
    print((df.groupby(col, observed=True)["is_returned"].mean() * 100).round(2))

age_group
18-25    5.47
26-35    5.39
36-45    5.93
46-55    5.40
56+      5.41
Name: is_returned, dtype: float64
customer_gender
Female    5.62
Male      5.44
Other     5.17
Name: is_returned, dtype: float64
payment_method
COD            5.10
Credit Card    5.55
Debit Card     5.67
PayPal         5.78
UPI            5.39
Wallet         5.38
Name: is_returned, dtype: float64
quantity
1    5.48
2    5.80
3    5.38
4    5.08
5    6.10
Name: is_returned, dtype: float64


In [70]:

# 4. Category ka profit %
print((df.groupby("category")["profit_margin"].sum() / df.groupby("category")["total_amount"].sum() * 100).round(1))

category
Beauty         32.2
Electronics    10.4
Fashion        27.3
Grocery       -11.2
Home           24.4
Sports         25.5
Toys           25.5
dtype: float64
